# **<center>03_extract_cnn_embeddings_for_ml</center>**

### Table of Contents

1. **Notebook Overview**  

2. **Environment Setup**  
   - Import of required libraries  

3. **Load Image and Metadata**  
   - Collect `.npy` image paths for the external test set  
   - Load and merge CSV metadata per subset
   - Basic integrity checks (images vs. metadata rows)  

4. **Metadata Extraction and Alignment**  
   - Extract labels and key variables (sex, birthdate, date_rx, station)  
   - Ensure 1:1 correspondence between `rx_cod`, images and metadata  

5. **Data Generators for CNN Feature Extraction**  
   - Custom `DataGenerator` for batched loading of `.npy` images  
   - Batch size, input shape and label encoding  

6. **Load Best CNN Model**  
   - Load the trained CNN feature extractor from disk  

7. **Deep Feature Extraction and Merge Features with Clinical / Technical Metadata** 
   - Build a `Model` to output convolutional feature maps  
   - Select the last convolutional block (VGG19 backbone)  
   - Apply global max pooling to obtain one feature vector per radiograph  
   - Iterative extraction over train and test generators  
   - Attach label and metadata (sex, age, dates, station name)  
   - Construct final train and test feature tables indexed by `rx_cod`  
   - Export feature matrices to CSV  

8. **Summary and Conclusions**  
   - Overview of the generated feature sets  
   - Role of these features in subsequent ML experiments  


### **1. Notebook Overview**

In this notebook we perform the deep feature extraction step required for the hybrid models. Instead of using the CNN only as an end-to-end classifier, we repurpose its internal representation to obtain one high-level feature vector per radiograph, which will later be used as input to classical machine-learning algorithms. The overarching objective is to fairly compare a pure CNN classifier with a hybrid approach that combines deep features and traditional ML classifiers on the same imaging data.

The analysis is based on external data from Institution 2. All radiographs are stored as preprocessed .npy arrays, and their corresponding metadata (e.g. labels, sex, dates, acquisition station) are loaded from curated CSV files.

Deep features are extracted from the VGG19 backbone of the best CNN model by taking the activations of the last convolutional block and applying global max pooling to obtain compact vectors that summarise the most salient image patterns. These features, combined with selected clinical and technical variables, form the basis for the subsequent hybrid Machine Learning external validadion test. 

### **2. Environment Setup**

In [30]:
# Standard libraries
import os
from datetime import datetime
from collections import Counter

# Numerical / data handling
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import GlobalMaxPooling2D
from tensorflow.keras.utils import Sequence, to_categorical


### **3. Load Images and Metadata**

In [2]:
# External generalization test subsets (Institution 2)

subset_external_test = [
    'TC_CZC2391BGR_test',
    'TC_CZC2161RCP_test',
    'TC_PRIMO_test'
]

subsets_external_test_name = 'external_test'
subsets_external_test_path = 'Dataset/External_text_Institution_2/Subsets_no_unet'


In [5]:
# Collect image paths and labels for EXTERNAL test set

images_external_test_path = []
labels_external_test = []

for subset in subset_external_test:

    subset_path = os.path.join(subsets_external_test_path, subset)
    csv_path = os.path.join(subsets_external_test_path, subset + '.csv')

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Missing CSV file for external subset: {csv_path}")

    df_ext = pd.read_csv(csv_path)
    label_dict_ext = dict(zip(df_ext['rx_cod'], df_ext['label_CalTend']))

    for img_name in os.listdir(subset_path):

        if not img_name.endswith('.npy'):
            continue

        img_path = os.path.join(subset_path, img_name)
        rx_cod = os.path.splitext(img_name)[0]

        if rx_cod in label_dict_ext:
            images_external_test_path.append(img_path)
            labels_external_test.append(int(label_dict_ext[rx_cod]))


In [6]:
print(f"External test images loaded: {len(images_external_test_path)}")
print("Distribución labels_external_test:", Counter(labels_external_test))

assert len(images_external_test_path) == len(labels_external_test), \
    "Mismatch in external test images/labels"


External test images loaded: 308
Distribución labels_external_test: Counter({0: 154, 1: 154})


In [8]:
# Load images from .npy files
def load_images(images_path):
    X_list = []
    for image_path in tqdm(images_path, desc='Cargando imágenes'):
        try:
            X_ = np.load(image_path)
            X_list.append(X_)
        except Exception as e:
            print(f' Error al cargar {image_path}: {e}')
    X = np.array(X_list)
    return X


In [13]:
X_external_test = load_images(images_external_test_path)
y_external_test = np.array(labels_external_test)

print("External test data shape:", X_external_test.shape)
print("External test label distribution:", Counter(y_external_test))


Cargando imágenes: 100%|██████████████████████| 308/308 [00:37<00:00,  8.17it/s]


External test data shape: (308, 512, 512, 3)
External test label distribution: Counter({0: 154, 1: 154})


### **4. Metadata Extraction and Alignment**

In [32]:
# Metadata extraction for feature-learning experiments

def extract_metadata(df, image_paths):
    """
    Extracts metadata (labels and patient information) corresponding to
    a list of image paths. Assumes that each image file name starts with
    the radiograph code 'rx_cod'.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing metadata for the dataset.
    image_paths : list of str
        List of full paths to .npy image files.

    Returns
    -------
    rx_codes : np.ndarray
        Radiograph identifiers extracted from filenames.
    y : np.ndarray
        Binary labels for calcific tendinopathy (0/1).
    sex : np.ndarray
        Patient sex.
    birthdate : np.ndarray
        Birth date for each radiograph.
    date_rx : np.ndarray
        Date of radiograph acquisition.
    """

    # --- Build lookup dictionaries for fast access ---
    label_dict     = dict(zip(df['rx_cod'], df['label_CalTend']))
    sex_dict       = dict(zip(df['rx_cod'], df['sex']))
    birthdate_dict = dict(zip(df['rx_cod'], df['birthdate']))
    date_rx_dict   = dict(zip(df['rx_cod'], df['date_rx']))

    rx_codes = []
    y = []
    sex = []
    birthdate = []
    date_rx = []

    for path in image_paths:
        code = os.path.splitext(os.path.basename(path))[0]
        rx_codes.append(code)

        try:
            y.append(label_dict[code])
            sex.append(sex_dict[code])
            birthdate.append(birthdate_dict[code])
            date_rx.append(date_rx_dict[code])
        except KeyError:
            raise KeyError(
                f"Radiograph code '{code}' not found in metadata table. "
                f"Check filename consistency and CSV integrity."
            )

    return (
        np.array(rx_codes),
        np.array(y),
        np.array(sex),
        np.array(birthdate),
        np.array(date_rx)
    )


In [17]:
# Load EXTERNAL TEST metadata (single DataFrame) 

df_external_list = []

for subset in subset_external_test:
    csv_path = os.path.join(subsets_external_test_path, f"{subset}.csv")

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Missing external CSV file: {csv_path}")

    df_tmp = pd.read_csv(csv_path)
    df_tmp["subset"] = subset  # optional tracking
    df_external_list.append(df_tmp)

df_external_test = pd.concat(df_external_list, ignore_index=True)

print("External test metadata loaded:", df_external_test.shape)
print("Label distribution (external test):")
print(df_external_test["label_CalTend"].value_counts(dropna=False))


External test metadata loaded: (308, 17)
Label distribution (external test):
label_CalTend
0    154
1    154
Name: count, dtype: int64


In [18]:
# ===== Metadata extraction for EXTERNAL generalization test =====

# Extract metadata for external test set
rx_cod_external, y_external, sex_external, birthdate_external, date_rx_external = extract_metadata(
    df_external_test,
    images_external_test_path
)

print(f"External test labels shape: {y_external.shape}")

# Ensure unique index for fast access
df_external_test = df_external_test.set_index('rx_cod')

# Create metadata dictionary (external)
metadata_external = df_external_test[
    ['sex', 'birthdate', 'date_rx', 'station_name']
].to_dict(orient='index')


External test labels shape: (308,)


### **5. Data Generator for CNN Feature Extraction**

In [19]:
# A custom DataGenerator is used to efficiently load .npy images in batches.
# This avoids storing thousands of radiographs in memory simultaneously and 
# ensures scalable training/feature extraction with limited RAM. It also 
# enables reproducible shuffling and supports both binary and multi-class labels.

class DataGenerator(Sequence):
    """
    Custom data generator for loading batches of .npy images and labels.
    Designed for CNN feature extraction prior to ML classification.

    Parameters
    ----------
    image_paths : list of str
        Full paths to .npy image files.
    labels : array-like
        Corresponding labels (binary or multi-class).
    batch_size : int
        Number of samples per batch.
    img_size : tuple(int, int)
        Expected image size (height, width).
    class_mode : str, optional
        "binary" or "categorical". Default is "binary".
    shuffle : bool, optional
        Whether to shuffle the dataset after each epoch.
    seed : int, optional
        Random seed for reproducibility.
    """

    def __init__(self, image_paths, labels, batch_size, img_size,
                 class_mode="binary", shuffle=True, seed=42):

        self.image_paths = image_paths
        self.labels = np.array(labels)
        self.batch_size = batch_size
        self.img_size = img_size
        self.class_mode = class_mode
        self.shuffle = shuffle
        self.rng = np.random.default_rng(seed)

        self.indexes = np.arange(len(self.image_paths))
        if self.shuffle:
            self.rng.shuffle(self.indexes)

    def __len__(self):
        """Number of batches per epoch."""
        return int(np.ceil(len(self.image_paths) / self.batch_size))

    def __getitem__(self, idx):
        """Generate one batch of data."""
        start = idx * self.batch_size
        end = (idx + 1) * self.batch_size
        batch_idxs = self.indexes[start:end]

        batch_paths = [self.image_paths[i] for i in batch_idxs]
        batch_labels = self.labels[batch_idxs]

        X, y = self._load_batch(batch_paths, batch_labels)
        return X, y

    def on_epoch_end(self):
        """Shuffle indexes after each epoch."""
        if self.shuffle:
            self.rng.shuffle(self.indexes)

    def _load_batch(self, batch_paths, batch_labels):
        """
        Load images from disk and prepare label array.

        Returns
        -------
        X : np.ndarray
            Batch of images, shape (batch_size, H, W, 3), float32.
        y : np.ndarray
            Batch of labels, binary or one-hot.
        """
        X = np.empty((len(batch_paths), self.img_size[0], self.img_size[1], 3),
                     dtype=np.float32)

        for i, path in enumerate(batch_paths):
            img = np.load(path).astype(np.float32)
            X[i] = img  # Assumes preprocessing was done earlier (normalised, resized)

        y = np.array(batch_labels)

        if self.class_mode == "categorical":
            n_classes = len(np.unique(self.labels))
            y = to_categorical(y, num_classes=n_classes)

        return X, y


In [21]:
# ===== Data generator for EXTERNAL generalization test =====

batch_size = 32
img_size = (512, 512)

external_test_generator = DataGenerator(
    image_paths=images_external_test_path,
    labels=labels_external_test,
    batch_size=batch_size,
    img_size=img_size,
    shuffle=False   # IMPORTANT: no shuffle for external evaluation
)

# Sanity check
X_batch, y_batch = external_test_generator[0]

print("✔ External test batch loaded successfully")
print(f"X_batch shape: {X_batch.shape}  (batch_size, H, W, C)")
print(f"y_batch shape: {y_batch.shape}")
print(f"Sample labels: {y_batch[:10]}")


✔ External test batch loaded successfully
X_batch shape: (32, 512, 512, 3)  (batch_size, H, W, C)
y_batch shape: (32,)
Sample labels: [0 0 1 1 0 1 1 0 0 0]


### **6. Load Best CNN Model**

In [23]:
# Path to the best CNN model
model_path = (
    'Models/all_models/best_cnn_model.h5'
)

# Load the trained model
model = load_model(model_path)

# Display model summary
print('EXPERIMENT 1 — Best CNN model:')
model.summary()


2026-02-06 15:07:25.973473: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-02-06 15:07:26.054696: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-02-06 15:07:26.057901: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysf

EXPERIMENT 1 — Best CNN model:
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vgg19 (Functional)          (None, 16, 16, 512)       20024384  
                                                                 
 global_max_pooling2d (Glob  (None, 512)               0         
 alMaxPooling2D)                                                 
                                                                 
 dense (Dense)               (None, 1)                 513       
                                                                 
Total params: 20024897 (76.39 MB)
Trainable params: 20024897 (76.39 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


### **7. Deep Feature Extraction and Merge Features with Clinical / Technical Metadata**

In [31]:
# ===== Feature extraction for EXTERNAL generalization test =====

# Create metadata dictionary for EXTERNAL test only
metadata_external = df_external_test[
    ['sex', 'birthdate', 'date_rx', 'station_name']
].to_dict(orient='index')

# Create the feature extractor from VGG19
conv_model = model.get_layer('vgg19')  # Ensure it exists
last_conv_layer = conv_model.get_layer('block5_conv4')
feature_extractor = Model(
    inputs=conv_model.input,
    outputs=last_conv_layer.output
)

# ===== Utility function: calculate age at radiograph =====

def calculate_age(birthdate_str, date_rx_str):
    try:
        birth = datetime.strptime(birthdate_str, '%Y-%m-%d')
        rx    = datetime.strptime(date_rx_str, '%Y-%m-%d')
        return (rx - birth).days // 365
    except:
        return np.nan


# Batch feature extraction function (unchanged logic)
def extract_features(generator, metadata_dict):
    feature_list = []
    label_list = []
    rx_cod_list = []
    sex_list = []
    birthdate_list = []
    date_rx_list = []
    station_rx_list = []

    for i in range(len(generator)):
        X_batch, y_batch = generator[i]

        # Extract feature maps and apply pooling
        feature_maps = feature_extractor.predict(X_batch, verbose=0)
        pooled_features = GlobalMaxPooling2D()(feature_maps).numpy()

        feature_list.append(pooled_features)
        label_list.append(y_batch)

        # Get rx_cod from the batch
        batch_indexes = generator.indexes[
            i * generator.batch_size : (i + 1) * generator.batch_size
        ].tolist()

        batch_rx_cod = [
            os.path.splitext(os.path.basename(generator.image_paths[j]))[0]
            for j in batch_indexes
        ]

        rx_cod_list.extend(batch_rx_cod)

        # Add metadata
        for cod in batch_rx_cod:
            info = metadata_dict.get(
                cod,
                {'sex': None, 'birthdate': None, 'date_rx': None, 'station_name': None}
            )
            sex_list.append(info['sex'])
            birthdate_list.append(info['birthdate'])
            date_rx_list.append(info['date_rx'])
            station_rx_list.append(info['station_name'])

    features = np.vstack(feature_list)
    labels = np.concatenate(label_list).flatten()

    return (
        features,
        labels,
        rx_cod_list,
        sex_list,
        birthdate_list,
        date_rx_list,
        station_rx_list
    )


# ---- Run feature extraction for EXTERNAL test ----
features_external, y_external, rx_cod_external, sex_external, birthdate_external, date_rx_external, station_rx_external = extract_features(
    external_test_generator,
    metadata_external
)

# Create DataFrame
n_features = features_external.shape[1]
column_names = [f'feature_{i+1}' for i in range(n_features)]

df_features_external = pd.DataFrame(
    features_external,
    columns=column_names,
    index=rx_cod_external
)


# Save CSV in the SAME folder as external input data
df_features_external.to_csv(
    os.path.join(
        subsets_external_test_path,
        'EXP_ML_external_test_features.csv'
    )
)

print("External test features saved.")
print("Shape:", df_features_external.shape)
print("Label distribution:")
print(df_features_external['label'].value_counts())


External test features saved.
Shape: (308, 518)
Label distribution:
label
0    154
1    154
Name: count, dtype: int64


### **8. Summary and Conclusions**

This notebook implements a deep feature–extraction pipeline for the external generalization test of the hybrid CNN + ML framework. External shoulder radiographs and their metadata are loaded, validated for consistency, and processed using batched data generators to ensure efficient and reproducible inference.

Deep convolutional features are extracted from the final block of the VGG19 backbone embedded in the trained CNN and reduced to a single feature vector per radiograph via global max pooling. These features are combined with selected clinical and technical variables to form the final feature table used to evaluate the hybrid model under an external generalization setting.